# Lab 07-01: Build Sequential, Parallel, and Loop Workflows

**Book chapter:** Chapter 07: Orchestrating Multi-Agent Systems  
**Estimated time:** 25 minutes

## Overview

Cymbal Retail's merchandising team writes a product page for every new item. In this
lab, you build a listing pipeline that gathers stock, price, and review facts in
parallel, summarizes them, and refines a product description in a writer and critic
loop.

## Learning Objectives

By the end of this lab, you will be able to:

- Run independent agents concurrently with ParallelAgent
- Chain steps in a fixed order with SequentialAgent
- Refine output with LoopAgent, exit_loop, and max_iterations
- Pass data between agents with output_key and instruction placeholders

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No Google Cloud resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Workflow agent | Runs its sub-agents | Typical use |
|---|---|---|
| **SequentialAgent** | One after another | Pipelines with dependent steps |
| **ParallelAgent** | At the same time | Independent lookups that can run together |
| **LoopAgent** | Repeatedly, until exit or a limit | Generate-and-refine tasks |

Workflow agents do not use a model to decide the order, so the structure is predictable,
while each step can still be an LLM agent.

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define the tools for three research agents

Each research agent will call one of these tools. In a real system, the tools would query inventory, pricing, and review services.

In [ ]:
import asyncio
import os
from concurrent.futures import ThreadPoolExecutor

from google.adk.agents import Agent, LoopAgent, ParallelAgent
from google.adk.agents import SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import exit_loop
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")


def check_inventory(product: str) -> dict:
    """Returns the stock level of a product online and in stores."""
    return {"product": product, "online": 140, "Denver": 12, "Boulder": 0}


def get_price(product: str) -> dict:
    """Returns the current price and any promotion for a product."""
    return {"product": product, "price": 189.0,
            "promotion": "10% off for Gold members"}


def get_reviews(product: str) -> dict:
    """Returns the average rating and common themes in reviews."""
    return {"product": product, "rating": 4.6,
            "themes": ["waterproof", "runs half a size small"]}

## Step 2: Gather facts in parallel, then summarize them

`researcher` creates an agent that calls one tool and saves its finding to session
state with `output_key`. A **ParallelAgent** runs the three researchers concurrently,
each writing to its own key. The summarizer reads the three keys through the
`{inventory}`, `{pricing}`, and `{reviews}` placeholders in its instruction.

In [ ]:
def researcher(name, tool, key):
    """Creates an agent that calls one tool and saves its finding."""
    return Agent(name=name, model=MODEL, tools=[tool], output_key=key,
                 instruction=f"Call {tool.__name__} for the product in the "
                             "request and report the result in one "
                             "sentence.")


research = ParallelAgent(name="product_research", sub_agents=[
    researcher("inventory_agent", check_inventory, "inventory"),
    researcher("pricing_agent", get_price, "pricing"),
    researcher("reviews_agent", get_reviews, "reviews"),
])
summarizer = Agent(
    name="summarizer", model=MODEL, output_key="facts",
    instruction="Combine these facts into three short bullet points.\n"
                "Inventory: {inventory}\nPricing: {pricing}\n"
                "Reviews: {reviews}")

## Step 3: Refine a product description in a writer and critic loop

A **LoopAgent** runs the writer and the critic repeatedly. The critic calls the
built-in `exit_loop` tool when the draft meets every rule, and `max_iterations=3`
guarantees that the loop stops even if it never does. The `?` in `{draft?}` and
`{feedback?}` leaves the placeholder empty on the first iteration.

A **SequentialAgent** then joins research, summary, and refinement into one pipeline.

In [ ]:
writer = Agent(
    name="writer", model=MODEL, output_key="draft",
    instruction="Write a product description of at most 60 words from "
                "these facts: {facts}\nIf there is a previous draft, "
                "improve it.\nPrevious draft: {draft?}\n"
                "Critic feedback: {feedback?}")
critic = Agent(
    name="critic", model=MODEL, tools=[exit_loop], output_key="feedback",
    instruction="Review this draft: {draft}\nIt must mention "
                "waterproofing, the sizing advice, and the Gold member "
                "promotion, in 60 words or fewer. If it meets every rule, "
                "call exit_loop. Otherwise, list exactly what to fix.")
refine = LoopAgent(name="description_refiner", sub_agents=[writer, critic],
                   max_iterations=3)

pipeline = SequentialAgent(name="listing_pipeline",
                           sub_agents=[research, summarizer, refine])

## Step 4: Run the pipeline and watch which agent produces each event

Run the pipeline. Each line shows the agent that produced an event and any tool it called.

In [ ]:
sessions = InMemorySessionService()
runner = Runner(app_name="cymbal", agent=pipeline, session_service=sessions,
                auto_create_session=True)
product = types.Part(text="Trailhead waterproof boot")
message = types.Content(role="user", parts=[product])
for event in runner.run(user_id="merchandiser", session_id="listing-1",
                        new_message=message):
    calls = ", ".join(c.name for c in event.get_function_calls())
    print(f"{event.author:16} {'calls ' + calls if calls else 'responds'}")

**What to look for:** the three research agents call their tools in the same step,
the summarizer runs after all three, and the writer and critic alternate until the
critic calls `exit_loop`.

## Step 5: Read the results that each step saved to session state

Every step saved its result to session state, which is how the agents passed information to each other.

In [ ]:
lookup = sessions.get_session(app_name="cymbal", user_id="merchandiser",
                              session_id="listing-1")
with ThreadPoolExecutor(1) as pool:  # Works in scripts and notebooks.
    state = pool.submit(asyncio.run, lookup).result().state
for key in ["inventory", "pricing", "reviews", "facts", "feedback"]:
    print(f"\n{key}: {state.get(key)}")
print(f"\nFinal description:\n{state['draft']}")

**Expected result:** values for `inventory`, `pricing`, `reviews`, `facts`, and `feedback`, followed by the final description.

## Step 6: Experiment

- In Step 3, make the critic stricter, for example by requiring at most 30 words, and rerun
  Steps 3 to 5. The loop runs more iterations, and stops at `max_iterations` if the critic
  is never satisfied.
- Change `max_iterations` to 1 and rerun Steps 3 to 5.

## Check Your Understanding

**1. Why does each research agent write to a different state key?**

<details><summary>Show answer</summary>

The branches run concurrently. Separate keys prevent one branch from overwriting another's result.

</details>

**2. What two things can end a LoopAgent?**

<details><summary>Show answer</summary>

A sub-agent signaling completion, here by calling exit_loop, or reaching max_iterations.

</details>

**3. Why is a workflow agent more predictable than a coordinator LLM?**

<details><summary>Show answer</summary>

The order of steps is fixed in code, so no model decides which agent runs next.

</details>

## Summary

You combined ParallelAgent, SequentialAgent, and LoopAgent into one pipeline and passed
results between agents through session state. This lab created no resources.